# Local or global: which duration curve to fit

`Distribution()` fits a duration curve, but which one? By default, `scope="local"`, each
typical day fits the duration curve of the days it represents. With `scope="global"`, the
typical days together fit the duration curve of the whole series.

Suppose a model sizes a solar-plus-battery system on each typical day. Then each typical day
needs the right amount of sun. In this tutorial we compare both scopes on two weeks of solar
irradiance and load: first on the two weeks as a whole, then on each typical day, and finally
on the amount of sun each typical day gets. By the end you will have seen that

* `global` fits the two-week duration curve better,
* `local` fits the days behind each typical day better, and
* `global` moves sun from one typical day to the other.

This is part 2 of five [duration-curve tutorials](../index.md#duration-curves). Each part stands on its own.

## 1  Two weeks, two typical days

We take **1–14 April 2010** from the example data: hourly solar irradiance (`GHI`) and
electricity `Load`. Two typical days for two weeks is a deliberately tight budget; it makes
the differences easy to see.

Every run in this tutorial uses the same settings, so we write them down once in a small
function. Only its `representation` argument will change. `preserve_column_means=False`
switches off tsam's rescaling step, so that we see the values exactly as the representation
produced them.

In [ ]:
import numpy as np
import pandas as pd
import plotly.io as pio

import tsam
from tsam import ClusterConfig, Distribution
from tsam.metrics import series_statistics
from tsam.plot import compare_series

pio.renderers.default = "notebook_connected"

raw = pd.read_csv("../../data/testdata.csv", index_col=0, parse_dates=True)
data = raw.loc["2010-04-01":"2010-04-14", ["GHI", "Load"]]
UNITS = {"GHI": "W/m²", "Load": "MW"}
COLORS = {
    "original": "#000000",
    "original days": "#000000",
    "local": "#009E73",
    "global": "#AD3977",
}


def aggregate(representation):
    return tsam.aggregate(
        data,
        n_clusters=2,
        period_duration="1D",
        cluster=ClusterConfig(method="hierarchical", representation=representation),
        preserve_column_means=False,
    )

## 2  The two weeks as a whole

Run both scopes and compare them with the duration curves of the whole two weeks.
`result.reconstructed` lays the typical days back out over the two weeks, one copy for every
day each of them represents:

In [ ]:
local = aggregate(Distribution())
whole = aggregate(Distribution(scope="global"))

compare_series(
    {"original": data, "local": local.reconstructed, "global": whole.reconstructed},
    mode="duration_curve",
    reference="original",
    units=UNITS,
    colors=COLORS,
    title="The two weeks as a whole",
).show()

Both follow the original closely; by eye they are hard to tell apart. tsam's duration-curve
error tells them apart. `rmse_duration` compares the sorted values in normalized units, and
lower is better:

In [ ]:
pd.DataFrame(
    {
        "local": local.accuracy.rmse_duration,
        "global": whole.accuracy.rmse_duration,
    }
).round(4)

In [ ]:
# Keeps the text true when the notebook runs in CI; hidden in the docs.
assert (whole.accuracy.rmse_duration < local.accuracy.rmse_duration).all()

`global` is better for both attributes. It fits exactly the curve we are looking at.

## 3  Each typical day on its own

Now look at each typical day separately, against the duration curve of the days it
represents. That curve pools all hours of those days and sorts them; it is what
`scope="local"` fits.

In [ ]:
days_of = {}
for cid, n_days in local.cluster_counts.items():
    member_days = np.flatnonzero(local.cluster_assignments == cid)
    days_of[cid] = pd.concat(
        [data.iloc[day * 24 : (day + 1) * 24] for day in member_days]
    )
    compare_series(
        {
            "original days": days_of[cid],
            "local": local.cluster_representatives.loc[cid],
            "global": whole.cluster_representatives.loc[cid],
        },
        mode="duration_curve",
        reference="original days",
        units=UNITS,
        colors=COLORS,
        title=f"The typical day for {n_days:.0f} days, against those days",
    ).show()

In [ ]:
# Keeps the text true when the notebook runs in CI; hidden in the docs.
nine = next(cid for cid, n in local.cluster_counts.items() if n == 9)
five = next(cid for cid, n in local.cluster_counts.items() if n == 5)
assert round(whole.cluster_representatives.loc[nine, "GHI"].max()) == 532
assert round(local.cluster_representatives.loc[nine, "GHI"].max()) == 628
assert days_of[nine]["GHI"].max() == 668

Look at the solar curves. `local` follows the days behind each typical day; `global` doesn't.
On the typical day for 9 days, the brightest `global` hour reaches only 532 W/m², while those
days reached 668 W/m² and `local` reaches 628 W/m².

## 4  How much sun does each typical day get?

Compare the average irradiance of each typical day with the average of the days it
represents:

In [ ]:
pd.DataFrame(
    {
        "days": pd.Series(local.cluster_counts).astype(int),
        "original days": {cid: days["GHI"].mean() for cid, days in days_of.items()},
        "local": local.cluster_representatives.groupby(level=0)["GHI"].mean(),
        "global": whole.cluster_representatives.groupby(level=0)["GHI"].mean(),
    }
).round(1)

In [ ]:
# Keeps the text true when the notebook runs in CI; hidden in the docs.
local_means = local.cluster_representatives.groupby(level=0).mean()
whole_means = whole.cluster_representatives.groupby(level=0).mean()
for cid, days in days_of.items():
    np.testing.assert_allclose(local_means.loc[cid], days.mean())
assert (
    round(100 * (1 - whole_means.loc[nine, "GHI"] / days_of[nine]["GHI"].mean())) == 8
)
assert (
    round(100 * (whole_means.loc[five, "GHI"] / days_of[five]["GHI"].mean() - 1)) == 11
)

`local` keeps the average irradiance of each typical day's days exactly. `global` gives the
typical day for 9 days 8% less sun and the typical day for 5 days 11% more. Over the two
weeks this evens out:

In [ ]:
series_statistics(
    {"original": data, "local": local.reconstructed, "global": whole.reconstructed}
).round(1)

In [ ]:
# Keeps the text true when the notebook runs in CI; hidden in the docs.
for attr in ["GHI", "Load"]:
    assert abs(whole.reconstructed[attr].mean() - data[attr].mean()) < 0.1
    np.testing.assert_allclose(local.reconstructed[attr].mean(), data[attr].mean())

Both keep the two-week means. A model that only adds up over the two weeks sees almost no
difference between the scopes. A model that works on each typical day, sizing a battery for
its sun, would see a darker 9-day typical day and a brighter 5-day typical day than really
happened.

## What you've learned

* `scope="local"`, the default, fits each typical day to the duration curve of the days it
  represents and keeps their means.
* `scope="global"` fits the duration curve of the whole series better, but it can move
  values, and with them energy, from one typical day to another.
* Which one is better depends on which duration curve your model uses: the whole series', or
  each typical day's.

Where to go next:

* [Part 1: Fitting a duration curve](01_fitting.ipynb) — how the fitted values are built.
* [Part 3: Keeping the extremes](03_extremes.ipynb) — keep the real highest and lowest values.
* [Part 4: Which values occur together](04_ordering.ipynb) — place the fitted values in time.
* [Part 5: Making peaks coincide on purpose](05_coincident_peaks.ipynb) — when pairing the
  highest values with each other is exactly what you want.